In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import display

In [ ]:
import tensorflow as tf
from keras.models import Model
from keras.layers import Dropout
from keras.layers import Input, Dense
from keras.callbacks import EarlyStopping
from keras.losses import MeanSquaredError, SparseCategoricalCrossentropy

In [ ]:
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import  StandardScaler
from sklearn.model_selection import train_test_split

In [ ]:
original_dataDF = pd.read_csv('C:/Users/andre/Downloads/train_data.csv')
train_dataDF = original_dataDF.copy()

In [ ]:
train_dataDF.drop('Unnamed: 0', axis=1, inplace=True)

In [ ]:
train_dataDF.head()

In [ ]:
train_dataDF['implant_size'] = train_dataDF['implant_size'].astype(int)

In [ ]:
labels = train_dataDF['implant_size']
features = train_dataDF.drop(['implant_size'], axis=1)

In [ ]:
features_train, features_test, labels_train, labels_test = train_test_split(features, labels, test_size=0.2, random_state=42)

In [ ]:
scaler = StandardScaler()
scaler.fit(features_train)
features_train_scaled = scaler.transform(features_train)
features_test_scaled = scaler.transform(features_test)

In [ ]:
labels_train = labels_train - 1
labels_test = labels_test - 1

In [ ]:
seed_value = 42

np.random.seed(seed_value)
tf.random.set_seed(seed_value)

In [ ]:
from sklearn.model_selection import KFold

# Define the number of folds for cross-validation
n_splits = 5  # You can adjust this number as needed

# Initialize KFold
kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

# Lists to store accuracy scores for each fold
accuracy_scores = []

# Iterate over each fold
for fold, (train_index, val_index) in enumerate(kf.split(features_train_scaled)):
    print(f'Fold {fold + 1}/{n_splits}')

    # Get train and validation data for this fold
    X_train_fold, X_val_fold = features_train_scaled[train_index], features_train_scaled[val_index]
    y_train_fold, y_val_fold = labels_train[train_index], labels_train[val_index]

    # Create a new instance of the model for each fold
    autoencoder = Model(inputs, [decoded, classifier])
    autoencoder.compile(optimizer='adam', 
                        loss=[MeanSquaredError(), SparseCategoricalCrossentropy()],
                        loss_weights=[0.5, 0.5])

    # Train the model
    history = autoencoder.fit(X_train_fold, [X_train_fold, y_train_fold], 
                              epochs=200, batch_size=32, 
                              validation_data=(X_val_fold, [X_val_fold, y_val_fold]),
                              callbacks=[early_stopping]) 

    # Evaluate the model on the validation data for this fold
    predictions = autoencoder.predict(X_val_fold)
    predicted_labels = np.argmax(predictions[1], axis=1)
    accuracy = accuracy_score(y_val_fold, predicted_labels)
    accuracy_scores.append(accuracy)

# Calculate and print the mean accuracy across all folds
mean_accuracy = np.mean(accuracy_scores)
print(f'Mean Accuracy: {mean_accuracy}')


In [ ]:
import pandas as pd
import plotly.graph_objects as go

# Prepare the data
data = {
    'Methods': ['EchoPWC-Net-us', 'c-PWC-Net-gray-usft', 'PWC-Net', 'MITT', 'Exp 6', 'Exp 7', 'Exp 8'],
    'ESAOTE': [0.07, 0.07, 0.12, 0.12, 0.08, 0.08, 0.07],
    'GE': [0.07, 0.08, 0.13, 0.12, 0.08, 0.12, 0.07],
    'HITACHI': [0.06, 0.06, 0.10, 0.08, 0.05, 0.07, 0.05],
    'PHILIPS': [0.06, 0.07, 0.10, 0.09, 0.06, 0.09, 0.06],
    'SAMSUNG': [0.05, 0.07, 0.10, 0.09, 0.06, 0.09, 0.06],
    'SIEMENS': [0.06, 0.04, 0.09, 0.07, 0.07, 0.09, 0.06],
    'TOSHIBA': [0.06, 0.06, 0.10, 0.09, 0.07, 0.10, 0.06],
   
}

df = pd.DataFrame(data)
df.set_index('Methods', inplace=True)

# Create the line plot with points
fig = go.Figure()

# Add traces for each column
for column in df.columns:
    fig.add_trace(go.Scatter(
        x=df.index,
        y=df[column],
        mode='lines+markers',
        name=column
    ))

# Update layout to set white background and customize labels
fig.update_layout(
    # title='Line Plot with Points',

)
fig.update_layout(
    width=600,  # Adjust width here to make it less wide
    height=400,  # Optional, adjust height if needed
)
fig.show()
# Save the plot as SVG
fig.write_image('line_plot_EPE_unprop.svg')


In [ ]:
import pandas as pd
import plotly.graph_objects as go

# Prepare the data with updated values
data = {
    'Methods': ['MITT', 'Exp 6', 'Exp 7', 'Exp 8'],
    'ESAOTE': [0.96, 1.14, 1.67, 1.13],
    'GE': [1.21, 1.26, 1.65, 1.03],
    'HITACHI': [0.80, 0.94, 1.50, 0.81],
    'PHILIPS': [0.83, 0.90, 1.28, 0.83],
    'SAMSUNG': [0.84, 1.08, 1.50, 1.06],
    'SIEMENS': [0.81, 1.20, 1.81, 1.11],
    'TOSHIBA': [0.91, 1.39, 2.25, 1.32]
}

# Create the DataFrame
df = pd.DataFrame(data)
df.set_index('Methods', inplace=True)

# Create the line plot with points
fig = go.Figure()

# Add traces for each column
for column in df.columns:
    fig.add_trace(go.Scatter(
        x=df.index,
        y=df[column],
        mode='lines+markers',
        name=column
    ))

# Update layout to set a narrower width
fig.update_layout(
    width=600,  # Adjust width to make it less wide
    height=400,  # Optional, adjust height if needed
)

# Show the plot
fig.show()

# Save the plot as SVG
fig.write_image('line_plot_EPE_prop.svg')


In [ ]:
# Re-importing numpy and matplotlib libraries to execute properly
import matplotlib.pyplot as plt
import numpy as np

# Data for the new box plot (ES values: mean ± std)
experiments_es = ['BASELINE', 'MITT', 'EXP 1', 'EXP 2', 'EXP 3', 'EXP 4', 'EXP 5', 'EXP 6']
means_es = [15.49, 14.30, 12.00, 8.03, 11.63, 12.13, 12.26, 12.52]
std_devs_es = [4.87, 4.72, 4.12, 2.12, 4.04, 4.28, 4.24, 4.40]

# Create data points for the boxplot using normal distribution (mean, std)
data_es = [np.random.normal(mean, std, 100) for mean, std in zip(means_es, std_devs_es)]

# Create the box plot
plt.figure(figsize=(10, 6))
plt.boxplot(data_es, patch_artist=True, boxprops=dict(facecolor="lightgreen"))

# Customizing the plot
plt.xticks(np.arange(1, len(experiments_es) + 1), experiments_es)
plt.title('ES Results for Different Experiments')
plt.ylabel('ES')

# Show the plot
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Data for the new box plot (EPE values: mean ± std)
experiments_epe = ['EXP 1', 'EXP 2', 'EXP 3', 'EXP 4', 'EXP 5', 'EXP 6']
means_epe = [1.38, 4.71, 1.30, 1.18, 1.17, 1.09]
std_devs_epe = [0.51, 0.43, 0.44, 0.35, 0.39, 0.36]

# Create data points for the boxplot using normal distribution (mean, std)
data_epe = [np.random.normal(mean, std, 100) for mean, std in zip(means_epe, std_devs_epe)]

# Create the box plot
plt.figure(figsize=(10, 6))
plt.boxplot(data_epe, patch_artist=True, boxprops=dict(facecolor="lightgreen"))

# Customizing the plot
plt.xticks(np.arange(1, len(experiments_epe) + 1), experiments_epe)
# plt.title('EPE Results for Different Experiments')
plt.ylabel('EPE')

# Show the plot
plt.show()


In [ ]:
import plotly.graph_objects as go

# Data for the experiments (Exp 6, Exp 7, Exp 8)
experiments = ['Exp 6', 'Exp 7', 'Exp 8']
ms_values = [10620, 0, 14400]
in_vivo_values = [11785, 13933, 13933]

# Create the bar plot
fig = go.Figure()

# Add bars for MS with reduced width
fig.add_trace(go.Bar(
    x=experiments,
    y=ms_values,
    name='MS',
    marker_color='blue',
    width=0.4  # Reduce width of bars
))

# Add bars for In vivo with reduced width
fig.add_trace(go.Bar(
    x=experiments,
    y=in_vivo_values,
    name='In vivo',
    marker_color='green',
    width=0.4  # Reduce width of bars
))

# Update layout to set a fixed range, white background, and customize labels
fig.update_layout(
    # title='Comparison of MS and In vivo for Experiments 6, 7, and 8',
    # xaxis_title='Experiments',
    # yaxis_title='Values',
    # plot_bgcolor='white',
    # paper_bgcolor='white',
    barmode='group',  # Group bars side by side
    legend_title='Dataset',
    # yaxis=dict(
    #     range=[0, 15000]  # Fix y-axis value range
    # ),
    xaxis=dict(
        tickvals=experiments,  # Ensure x-axis labels are correct
        ticktext=experiments  # Ensure x-axis labels are correct
    )
)
fig.update_layout(
    width=600,  # Adjust width to make it less wide
    height=400,  # Optional, adjust height if needed
)
fig.show()
fig.write_image('barplot_SUS.svg')

In [ ]:
import plotly.graph_objects as go

# Data for the experiments (Exp 1 to Exp 6)
experiments = ['Exp 1', 'Exp 2', 'Exp 3', 'Exp 4', 'Exp 5', 'Exp 6']
ms_values = [0, 10620, 2645, 5292, 7937, 10620]
in_vivo_values = [11785, 0, 11785, 11785, 11785, 11785]

# Create the bar plot
fig = go.Figure()

# Add bars for MS
fig.add_trace(go.Bar(
    x=experiments,
    y=ms_values,
    name='MS',
    marker_color='blue',
    width=0.4  # Thicker bars
))

# Add bars for In vivo
fig.add_trace(go.Bar(
    x=experiments,
    y=in_vivo_values,
    name='In vivo',
    marker_color='green',
    width=0.4  # Thicker bars
))

# Update layout to set a fixed range, white background, and customize labels
fig.update_layout(
    # title='Comparison of MS and In vivo for Experiments 1 to 6',
    # xaxis_title='Experiments',
    # yaxis_title='Values',
    # plot_bgcolor='white',
    # paper_bgcolor='white',
    barmode='group',  # Group bars side by side
    legend_title='Dataset',
    yaxis=dict(
        range=[0, 15000]  # Fix y-axis value range
    ),
    # xaxis=dict(
    #     tickvals=experiments,  # Ensure x-axis labels are correct
    #     ticktext=experiments  # Ensure x-axis labels are correct
    # )
)

# Show the plot
fig.show()
fig.write_image('barplot_In_vivoMS.svg')